<a href="https://colab.research.google.com/github/hossain8078/Machine-Learning-and-Deep-Learning/blob/main/05_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. Automated Machine Learning Pipelines

This notebook provides an enhanced exploration of **Automated Machine Learning Pipelines**. Pipelines are a powerful concept in machine learning that allow you to chain together multiple data processing and modeling steps into a single, cohesive workflow. This significantly improves the organization, reproducibility, and maintainability of your machine learning projects.

**What is an ML Pipeline?**
At its core, a machine learning pipeline is a sequence of data transformations and a final estimator (model). The output of one step serves as the input for the next. Think of it as an assembly line for your data, where each station performs a specific task.

**Why are Pipelines Crucial?**
-   **Streamlined Workflow:** Simplifies complex sequences of operations into a single object, making your code cleaner and easier to understand.
-   **Consistency:** Ensures that all data (training, validation, and new unseen data) undergoes the exact same sequence of transformations, preventing inconsistencies.
-   **Prevents Data Leakage:** This is perhaps the most critical benefit. Pipelines ensure that data transformations (like scaling or imputation) learned from the training data are applied *only* to the training data, and then consistently to the test data. Without pipelines, it's easy to accidentally 'leak' information from your test set into your training process, leading to overly optimistic performance estimates.
-   **Reproducibility:** A pipeline encapsulates your entire workflow, making it easy to reproduce your results and share your methodology with others.
-   **Modularity:** Each step in the pipeline is a distinct component, allowing for easy swapping or modification of individual steps without affecting the entire structure.
-   **Hyperparameter Tuning:** Enables tuning of hyperparameters across multiple steps of the workflow simultaneously, leading to better overall model performance.

We will build a comprehensive pipeline that includes preprocessing, feature selection, and a classifier, and then demonstrate how to tune the entire pipeline using `GridSearchCV`.

### Step 1: Import Libraries

Let's import all the necessary libraries for data manipulation, preprocessing, model building, and hyperparameter tuning.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris, load_wine
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6) # Set default figure size

### Step 2: Load and Prepare the Dataset

We will use the **Wine dataset** for this example. It's a multi-class classification problem with numerical features, making it suitable for demonstrating preprocessing and classification within a pipeline.

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

print("--- Dataset Head ---")
print(X.head())

print("--- Dataset Info ---")
X.info()

print("Target distribution:")
print(pd.Series(y).value_counts())

--- Dataset Head ---
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline 

### Step 3: Define Preprocessing Steps with `ColumnTransformer`

Even though the Wine dataset is all numerical, we'll set up a `ColumnTransformer` to demonstrate its usage. This is particularly useful when you have mixed data types (numerical and categorical) and need to apply different transformations to different columns.

In [ ]:
# For this dataset, all features are numerical and will be scaled.
numerical_features = X.columns.tolist()

# Define a transformer for numerical features: Impute (if any missing) and then Scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # In case there were missing values
    ('scaler', StandardScaler())
])

# Create the ColumnTransformer. 'remainder='passthrough'' means non-specified columns are kept.
# 'remainder='drop'' would drop them.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features)
    ],
    remainder='passthrough'
)

print("ColumnTransformer for preprocessing created!")

### Step 4: Create the Full Machine Learning Pipeline

Now, we'll combine our `preprocessor` with a feature selection step (`SelectKBest`) and a final classifier (`SVC`) into a single `Pipeline` object. This represents our entire machine learning workflow.

In [ ]:
# Create the full pipeline
# Steps: Preprocessing -> Feature Selection -> Classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(f_classif, k=8)), # Select top 8 features based on ANOVA F-value
    ('classifier', SVC(random_state=42))
])

print("Full ML Pipeline created!")

### Step 5: Split Data and Train the Pipeline

We split our data into training and testing sets and then train the entire pipeline on the training data. When you call `.fit()` on the pipeline, it sequentially calls `fit_transform()` on all transformers (except the last one) and then `fit()` on the final estimator.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training pipeline...")
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

### Step 6: Evaluate the Pipeline

After training, we evaluate the pipeline's performance on the unseen test set. Calling `.score()` or `.predict()` on the pipeline automatically applies all the defined preprocessing and feature selection steps before making predictions.

In [ ]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Pipeline Accuracy on Test Set: {accuracy:.4f}")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
# Disable the gridlines to prevent them from rendering over the heatmap
plt.grid(False)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=wine.target_names, yticklabels=wine.target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix for Pipeline Prediction")
plt.show()

### Step 7: Hyperparameter Tuning with a Pipeline

One of the most powerful features of pipelines is their seamless integration with hyperparameter tuning tools like `GridSearchCV`. You can define a search space that includes hyperparameters from *any* step in the pipeline.

In [ ]:
# Define the parameter grid for GridSearchCV
# Note the 'stepname__parameter' syntax
param_grid = {
    'selector__k': [5, 8, 10, 'all'], # Number of features to select
    'classifier__C': [0.1, 1, 10], # SVC regularization parameter
    'classifier__kernel': ['rbf', 'linear'] # SVC kernel type
}

# Create the GridSearchCV object
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the grid search to the training data
print("Starting Grid Search on Pipeline...")
grid_search.fit(X_train, y_train)

print("Grid Search on Pipeline complete!")

In [ ]:
print(f"Best parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Evaluate the best pipeline on the test set
best_pipeline = grid_search.best_estimator_
y_pred_tuned = best_pipeline.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Accuracy of Tuned Pipeline on Test Set: {accuracy_tuned:.4f}")

print("Classification Report for Tuned Pipeline:")
print(classification_report(y_test, y_pred_tuned, target_names=wine.target_names))

## Expanding on Data Leakage

Data leakage is a critical issue in machine learning where information from outside the training data is used to create the model, leading to overly optimistic performance estimates on the training set and poor generalization to new, unseen data.

### Types of Data Leakage:

1.  **Target Leakage:** Occurs when the features used to train the model directly or indirectly contain information about the target variable that would not be available at prediction time.
    *   **Example:** Predicting if a patient has a disease using a feature like `treatment_received` that is only recorded *after* the disease diagnosis.

2.  **Train-Test Contamination:** Occurs when information from the test set 'leaks' into the training process. This is the most common form prevented by pipelines.
    *   **Example:** Scaling features using the `mean` and `standard deviation` calculated from the *entire dataset* (training + test) before splitting, instead of fitting the scaler only on the training data.

### How Pipelines Prevent Leakage:

`sklearn.pipeline.Pipeline` is designed to prevent train-test contamination by ensuring that all data transformations (like imputation, scaling, feature selection, etc.) are fitted *only* on the training data. The learned transformations are then applied consistently to both the training and test data (and eventually to new, unseen data). When you call `pipeline.fit(X_train, y_train)`, each transformer's `fit_transform` method is called sequentially. When you call `pipeline.predict(X_test)` or `pipeline.score(X_test, y_test)`, each transformer's `transform` method is called.

## Handling Imbalanced Data in Pipelines

Many real-world datasets suffer from class imbalance, where one class significantly outnumbers others. This can lead to models that perform well on the majority class but poorly on the minority class. Techniques like oversampling (e.g., SMOTE) or undersampling can address this, and it's crucial to integrate them correctly into a pipeline to prevent data leakage.

**Key Principle:** Resampling techniques (especially oversampling like SMOTE) should *only* be applied to the training data. Applying them before the train-test split would lead to synthetic samples in the test set, creating an unrealistic evaluation.

The `imblearn` (imbalanced-learn) library provides `Pipeline` compatible transformers for various resampling strategies.

In [ ]:
# First, let's install the imbalanced-learn library if you haven't already
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # Use imblearn's Pipeline for resampling steps
from sklearn.linear_model import LogisticRegression

# Create a dummy imbalanced dataset for demonstration
# Let's assume we have an imbalanced version of the wine dataset
# For simplicity, we'll artificially create imbalance in a copy
X_imb = X.copy()
y_imb = y.copy()

# Artificially reduce the number of samples for class 2 to simulate imbalance
class_2_indices = np.where(y_imb == 2)[0]
np.random.seed(42)
delete_indices = np.random.choice(class_2_indices, size=int(len(class_2_indices) * 0.7), replace=False)

X_imb = X_imb.drop(delete_indices).reset_index(drop=True)
y_imb = np.delete(y_imb, delete_indices)

print("Original target distribution:")
print(pd.Series(y).value_counts())
print("\nImbalanced target distribution:")
print(pd.Series(y_imb).value_counts())

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)

# Define the preprocessing steps (same as before)
preprocessor_imb = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features)
    ],
    remainder='passthrough'
)

# Create an imbalanced-learn pipeline that includes SMOTE
# Note: SMOTE should be applied AFTER preprocessing and BEFORE the classifier
# It generates synthetic samples in the feature space, so scaling should happen first.
imb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor_imb),
    ('smote', SMOTE(random_state=42)), # Oversampling step
    ('classifier', LogisticRegression(random_state=42, solver='liblinear')) # Using Logistic Regression for faster training
])

print("\nTraining imbalanced pipeline with SMOTE...")
imb_pipeline.fit(X_train_imb, y_train_imb)

print("Imbalanced pipeline trained successfully!")

y_pred_imb = imb_pipeline.predict(X_test_imb)
accuracy_imb = accuracy_score(y_test_imb, y_pred_imb)
print(f"\nAccuracy of Imbalanced Pipeline on Test Set: {accuracy_imb:.4f}")
print("Classification Report for Imbalanced Pipeline:")
print(classification_report(y_test_imb, y_pred_imb))


## Model Persistence (Saving and Loading Pipelines)

Once you have trained and tuned a machine learning pipeline, you'll often want to save it to disk and load it later to make predictions on new data without retraining. This is crucial for deploying your models in production environments.

Python's `joblib` library (part of `scikit-learn`'s dependencies) is highly recommended for saving and loading `numpy` arrays and `scikit-learn` objects, as it's more efficient than `pickle` for large arrays.

**Steps:**
1.  **Save:** Use `joblib.dump()` to serialize your trained pipeline to a file.
2.  **Load:** Use `joblib.load()` to deserialize the pipeline back into memory.

In [ ]:
import joblib
import os

# Let's save the 'best_pipeline' obtained from our previous GridSearchCV
# (assuming `best_pipeline` is available from the last run)

# If best_pipeline is not available, you can use the 'pipeline' defined earlier
# For this example, let's ensure we have a trained pipeline
if 'best_pipeline' not in locals():
    print("Best pipeline not found, using the initial 'pipeline' and fitting it.")
    # Re-create and train a basic pipeline for demonstration if best_pipeline isn't defined
    pipeline_to_save = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('selector', SelectKBest(f_classif, k=8)),
        ('classifier', SVC(random_state=42))
    ])
    pipeline_to_save.fit(X_train, y_train)
else:
    pipeline_to_save = best_pipeline


# Define a filename for the saved pipeline
pipeline_filename = 'trained_ml_pipeline.joblib'

# Save the pipeline
joblib.dump(pipeline_to_save, pipeline_filename)
print(f"Pipeline saved successfully to '{pipeline_filename}'")

# --- Now, let's simulate loading the pipeline in a new session ---
print(f"\nLoading pipeline from '{pipeline_filename}'...")
loaded_pipeline = joblib.load(pipeline_filename)
print("Pipeline loaded successfully!")

# Verify by making a prediction with the loaded pipeline
# We'll use a single sample from the test set for demonstration
sample_X = X_test.iloc[[0]]
sample_y_true = y_test[0]

predicted_class = loaded_pipeline.predict(sample_X)[0]

print(f"\nOriginal sample true label: {wine.target_names[sample_y_true]}")
print(f"Loaded pipeline predicted label: {wine.target_names[predicted_class]}")

# Clean up the file (optional)
# os.remove(pipeline_filename)
# print(f"Cleaned up '{pipeline_filename}'")


## Visualizing Pipelines

Understanding the structure of complex pipelines, especially those involving `ColumnTransformer`, can be challenging. `scikit-learn` provides built-in tools to visualize pipelines, making them easier to debug and explain.

Since `scikit-learn` version 0.23, you can use `set_config(display='diagram')` to render pipelines as HTML diagrams in Jupyter environments, which is very helpful for a walkthrough.

In [ ]:
from sklearn import set_config

# Set the display configuration to 'diagram'
set_config(display='diagram')

print("Displaying the structure of our 'pipeline' object:")
# Display the pipeline (this will render an interactive diagram if in a rich environment like Colab/Jupyter)
display(pipeline)

# You can also display sub-components, like the preprocessor
print("\nDisplaying the structure of the 'preprocessor' object:")
display(preprocessor)

# Remember to reset if you prefer the default text representation
# set_config(display='text')


## Pipeline with Custom Transformers

Sometimes, `scikit-learn`'s built-in transformers aren't sufficient for your specific preprocessing or feature engineering needs. In such cases, you can create custom transformers that seamlessly integrate into `sklearn` pipelines.

To create a custom transformer, your class must inherit from:
-   `BaseEstimator`: Provides `get_params` and `set_params` methods, which are essential for `GridSearchCV`.
-   `TransformerMixin`: Provides the `fit_transform` method, which is convenient.

You must implement at least:
-   `__init__(self, ...)`: Constructor to set hyperparameters.
-   `fit(self, X, y=None)`: Learn transformation parameters from `X`. Should return `self`.
-   `transform(self, X)`: Apply the transformation to `X`.

Let's create a simple custom transformer that adds a new feature by combining existing ones.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# Define a custom transformer
class RatioFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, feature1_col, feature2_col, new_feature_name='ratio_feature'):
        self.feature1_col = feature1_col
        self.feature2_col = feature2_col
        self.new_feature_name = new_feature_name

    def fit(self, X, y=None):
        # Nothing to learn here, so just return self
        return self

    def transform(self, X):
        X_copy = X.copy() # Work on a copy to avoid modifying the original DataFrame
        # Check if features exist and are not zero to avoid division by zero
        if self.feature1_col in X_copy.columns and self.feature2_col in X_copy.columns:
            # Handle potential division by zero by replacing zeros with a small epsilon or NaN
            X_copy[self.new_feature_name] = X_copy[self.feature1_col] / (X_copy[self.feature2_col].replace(0, 1e-6))
        else:
            print(f"Warning: One or both features '{self.feature1_col}', '{self.feature2_col}' not found. Custom feature not created.")
            X_copy[self.new_feature_name] = np.nan # Add NaN column if features are missing
        return X_copy

# Create a new pipeline including our custom transformer
# Note: Custom transformers that add or remove columns should be placed carefully
# Often, they come before a ColumnTransformer if they operate on raw features.
# Or, they can be part of a custom preprocessor that handles column names.

# For simplicity, let's put it at the start of a new pipeline, assuming it works with DataFrame output
# and that numerical_features will be updated for the ColumnTransformer.

# Let's define an updated list of numerical features that includes our potential new feature
# For the wine dataset, let's create a ratio of 'flavanoids' to 'total_phenols'
custom_numerical_features = numerical_features + ['flavanoids_to_total_phenols']

custom_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, custom_numerical_features) # Use updated list here
    ],
    remainder='passthrough'
)

custom_pipeline = Pipeline(steps=[
    ('feature_engineer', RatioFeatureEngineer(feature1_col='flavanoids', feature2_col='total_phenols', new_feature_name='flavanoids_to_total_phenols')),
    ('preprocessor', preprocessor), # Re-using the original preprocessor for now, but in a real case, it might need to know about the new feature.
    ('classifier', RandomForestClassifier(random_state=42))
])

# To make the preprocessor truly aware of the new feature, we need to pass the updated list to ColumnTransformer
# Let's reconstruct preprocessor for this pipeline to include the new feature:
custom_preprocessor_with_new_feature = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, custom_numerical_features)
    ],
    remainder='passthrough'
)

custom_pipeline_updated = Pipeline(steps=[
    ('feature_engineer', RatioFeatureEngineer(feature1_col='flavanoids', feature2_col='total_phenols', new_feature_name='flavanoids_to_total_phenols')),
    ('preprocessor', custom_preprocessor_with_new_feature),
    ('selector', SelectKBest(f_classif, k='all')),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("\nTraining pipeline with custom transformer...")
custom_pipeline_updated.fit(X_train, y_train)

y_pred_custom = custom_pipeline_updated.predict(X_test)
accuracy_custom = accuracy_score(y_test, y_pred_custom)
print(f"Accuracy of Custom Pipeline on Test Set: {accuracy_custom:.4f}")
print("Classification Report for Custom Pipeline:")
print(classification_report(y_test, y_pred_custom))


## Comparison: Pipelines vs. Manual Steps

To truly appreciate the value of `sklearn.pipeline.Pipeline`, it's helpful to consider the alternative: performing all preprocessing, feature engineering, and modeling steps manually.

### Manual Steps: The Pitfalls

1.  **Code Repetition & Complexity:** You'd have to write separate code for scaling `X_train`, then `X_test`, then new unseen data. This quickly becomes repetitive and error-prone.
2.  **Increased Risk of Data Leakage:** Manually applying transformations opens the door to accidental leakage. For instance, fitting a `StandardScaler` on `X_test` by mistake, or applying feature selection before splitting the data, would inflate performance metrics.
3.  **Lack of Reproducibility:** Without a single, cohesive workflow, reproducing results accurately becomes difficult. Tiny discrepancies in the order or application of steps can lead to different outcomes.
4.  **Maintenance Headaches:** Modifying a step (e.g., changing the imputer strategy) requires updating code in multiple places, increasing the chance of introducing bugs.
5.  **Difficult Hyperparameter Tuning:** Tuning parameters across different stages (e.g., `k` in `SelectKBest` and `C` in `SVC`) would require nested loops and complex bookkeeping, which `GridSearchCV` and pipelines handle effortlessly.

### Pipelines: The Solution

-   **Encapsulation:** All steps are bundled into a single object.
-   **Automation:** `fit()`, `transform()`, and `predict()` methods are automatically called in sequence.
-   **Guaranteed Consistency & No Leakage:** Transformations are fit on training data and applied to all data consistently.
-   **Simplified Hyperparameter Tuning:** `GridSearchCV` (or `RandomizedSearchCV`) can tune parameters for *any* step in the pipeline using a simple `step_name__param_name` syntax.
-   **Readability & Maintainability:** Code is cleaner and easier to manage.

In essence, pipelines elevate your machine learning workflow from a fragmented script to a robust, production-ready system.

## Ensembling with Pipelines

Ensemble methods combine predictions from multiple base estimators to improve overall accuracy and robustness. `scikit-learn` provides several ensemble techniques like `VotingClassifier` (for classification) and `StackingClassifier`. These can be seamlessly integrated into a pipeline.

### `VotingClassifier`

Combines predictions from multiple diverse base estimators. It can use a 'hard' voting scheme (majority class label) or a 'soft' voting scheme (average of predicted probabilities).

### `StackingClassifier`

Trains a meta-model (final estimator) to combine the predictions of several base estimators. The base estimators are trained on the full training data, and the meta-model is trained on the predictions of the base estimators.

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Let's define a preprocessor (can reuse the one from earlier)
# We'll use the preprocessor that scales numerical features

# Define individual classifiers
clf1 = LogisticRegression(random_state=42, solver='liblinear')
clf2 = RandomForestClassifier(random_state=42)
clf3 = SVC(random_state=42, probability=True) # SVC needs probability=True for 'soft' voting

# Create a VotingClassifier with different base estimators
# 'soft' voting uses predicted probabilities (requires base estimators to support predict_proba)
ensemble_classifier = VotingClassifier(estimators=[
    ('lr', clf1),
    ('rf', clf2),
    ('svc', clf3)
], voting='soft', n_jobs=-1)

# Create a pipeline with the ensemble classifier
ensemble_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(f_classif, k='all')),
    ('ensemble_clf', ensemble_classifier)
])

# Train the ensemble pipeline
print("\nTraining ensemble pipeline...")
ensemble_pipeline.fit(X_train, y_train)

print("Ensemble pipeline trained successfully!")

# Evaluate the ensemble pipeline
y_pred_ensemble = ensemble_pipeline.predict(X_test)
accuracy_ensemble = accuracy_score(y_test, y_pred_ensemble)
print(f"\nAccuracy of Ensemble Pipeline on Test Set: {accuracy_ensemble:.4f}")
print("Classification Report for Ensemble Pipeline:")
print(classification_report(y_test, y_pred_ensemble, target_names=wine.target_names))

# You can also tune the ensemble pipeline with GridSearchCV
# Example param_grid (be careful, this can be computationally intensive!):
# param_grid_ensemble = {
#     'ensemble_clf__lr__C': [0.1, 1],
#     'ensemble_clf__rf__n_estimators': [50, 100],
#     'ensemble_clf__voting': ['hard', 'soft']
# }
# grid_search_ensemble = GridSearchCV(ensemble_pipeline, param_grid_ensemble, cv=3, n_jobs=-1)
# grid_search_ensemble.fit(X_train, y_train)
# print(f"Best ensemble params: {grid_search_ensemble.best_params_}")


### Mini-Challenges / Further Exploration

1.  **Add a Different Classifier:** Replace `SVC` with another classifier like `RandomForestClassifier` or `LogisticRegression` in the pipeline. Adjust the `param_grid` to include relevant hyperparameters for the new classifier and re-run the grid search.
2.  **Experiment with Feature Selection:** Change `SelectKBest` to `PCA` (Principal Component Analysis) for dimensionality reduction. You'll need to import `PCA` from `sklearn.decomposition`. How does `n_components` in `PCA` affect performance?
    ```python
    from sklearn.decomposition import PCA
    # pipeline = Pipeline(steps=[..., ('pca', PCA(n_components=2)), ...])
    # param_grid = {..., 'pca__n_components': [2, 3, 4], ...}
    ```
3.  **Custom Preprocessing Step:** Create a custom transformer (by inheriting from `BaseEstimator` and `TransformerMixin`) that performs a specific data cleaning or feature engineering task (e.g., removing outliers, creating interaction terms). Integrate this custom transformer into your pipeline.
4.  **Pipeline for Regression:** Adapt this notebook to build a pipeline for a regression problem (e.g., using the California Housing dataset). What preprocessing steps and regression models would you include? How would you evaluate the performance?